<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Attention_and_Prompted_probes_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

# Setup files

Downloading necessary modules

In [2]:
import transformer_lens
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


import numpy as np
import pandas as pd
import os
import json
import requests
from pathlib import Path
from typing import List, Dict
from typing import Optional, Literal
from collections import Counter
import random
import gc

import plotly.express as px
import matplotlib

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

2025-12-27 09:24:29.848160: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766827470.039837     129 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766827470.095016     129 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766827470.548201     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766827470.548249     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766827470.548253     129 computation_placer.cc:177] computation placer alr

## Downloading the Model

In [3]:
model = transformer_lens.HookedTransformer.from_pretrained("Qwen/Qwen2.5-0.5B")



config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


## Downloading the Train and Test datasets

In [4]:
DATA_DIR = Path("data/high_stakes")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/training/prompts_4x/train.jsonl"
train_path = DATA_DIR / "train.jsonl"

response = requests.get(train_url)
response.raise_for_status()

train_path.write_bytes(response.content)
print("Saved train data to", train_path)


MT_url = "https://pub-fd16e959a4f14ca48765b437c9425ba6.r2.dev/evals/dev/mt_balanced_apr_30.jsonl"
MT_dev_path = DATA_DIR / "MT_dev.jsonl"


response = requests.get(MT_url)
response.raise_for_status()
MT_dev_path.write_bytes(response.content)

print('Saved test data to', MT_dev_path)


Saved train data to data/high_stakes/train.jsonl
Saved test data to data/high_stakes/MT_dev.jsonl


In [5]:
def load_jsonl(path) -> List[Dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

def label_to_int(x: str) -> int:
    if x == "high-stakes":
        return 1
    elif x == "low-stakes":
        return 0
    else:
        raise ValueError(f"Unexpected label: {x!r}")


def normalize_inputs(inputs_field: str) -> str:
    s = inputs_field.strip()


    if s.startswith('[') and '"role"' in s:
        try:
            messages = json.loads(s)
            parts = [f"{m['role']}: {m['content']}" for m in messages]
            return "\n".join(parts)
        except json.JSONDecodeError:

            return inputs_field
    else:

        return inputs_field


In [6]:
train_rows = load_jsonl("data/high_stakes/train.jsonl")
dev_rows   = load_jsonl("data/high_stakes/MT_dev.jsonl")
len(train_rows), len(dev_rows)

(8000, 278)

In [7]:
train_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in train_rows]
test_dataset = [{'text': normalize_inputs(row['inputs']), 'label': label_to_int(row['labels'])} for row in dev_rows]

train_texts = [train['text'] for train in train_dataset]
train_labels= [train['label'] for train in train_dataset]

test_texts = [test['text'] for test in test_dataset]
test_labels= [test['label'] for test in test_dataset]


combined = list(zip(train_texts, train_labels))

random.shuffle(combined)

train_texts, train_labels = zip(*combined)
train_texts = list(train_texts)
train_labels = list(train_labels)

In [8]:
def create_dataloaders(
    activations: np.ndarray,
    labels: List[int],
    batch_size: int = 32,
    train_split: float = 0.8
):
    """Create train/val dataloaders from activations and labels"""

    # Convert to tensors
    X = torch.FloatTensor(activations)
    y = torch.FloatTensor(labels)

    # Create dataset
    dataset = TensorDataset(X, y)

    # Split train/val if needed
    if train_split < 1.0:
        train_size = int(train_split * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(
            dataset, [train_size, val_size]
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        return train_loader, val_loader
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        return loader


In [9]:
def get_activations(texts, model, layer_idx=-1, batch_size=8, pooling='last', pad_all=True):
    """
    Extract activations with different pooling strategies.

    Args:
        pad_all: If True and pooling='all', pad sequences to same length
    """
    model.eval()
    all_activations = []

    if layer_idx < 0:
      hook_name = f'blocks.{model.cfg.n_layers+layer_idx}.hook_resid_post'
    else:
      hook_name = f'blocks.{layer_idx}.hook_resid_post'

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        for text in batch_texts:
            captured = []

            def hook_fn(activation, hook):
                captured.append(activation.clone().cpu())

            with torch.no_grad():
                model.run_with_hooks(
                    text,
                    fwd_hooks=[(hook_name, hook_fn)]
                )

            hidden_states = captured[0][0]  # [seq_len, d_model]

            if pooling == 'last':
                act = hidden_states[-1, :]
            elif pooling == 'mean':
                act = hidden_states.mean(dim=0)
            elif pooling == 'first':
                act = hidden_states[0, :]
            elif pooling == 'all':
                act = hidden_states  # [seq_len, d_model]

            all_activations.append(act)
            del captured, hidden_states, act

        torch.cuda.empty_cache()

    # Concatenate based on pooling
    if pooling == 'all':
        if pad_all:
            # Pad to same length
            from torch.nn.utils.rnn import pad_sequence
            padded = pad_sequence(all_activations, batch_first=True)
            return padded.numpy()  # [num_texts, max_seq_len, d_model]
        else:
            return all_activations  # List of varying length tensors
    else:
        return torch.stack(all_activations, dim=0).numpy()

In [10]:
class LinearProbe(nn.Module):


    def __init__(self, d_model: int, n_classes: int = 1):
        super().__init__()

        self.linear = nn.Linear(d_model, n_classes, bias = True)

    def forward(self, activations: torch.Tensor) -> torch.Tensor:
        """
        Args:
            activations: shape [batch, seq_len, d_model] OR [batch, d_model]
        Returns:
            logits: shape [batch, n_classes]
        """
        return self.linear(activations)



class AttentionProbe(nn.Module):
    def __init__(self, d_model: int):
        """
        d_model: hidden size of the LM layer you’re probing.
        """
        super().__init__()

        self.q = nn.Linear(d_model, 1, bias=True)
        self.classifier = nn.Linear(d_model, 1, bias=True)


    def forward(self, x):
            """
            x: [batch, seq_len, d_model]
            Returns: logits [batch, 1]
            """
            scores = self.q(x).squeeze(-1)      # [B, T]
            attn = F.softmax(scores, dim=-1)               # [B, T]
            pooled = (attn.unsqueeze(-1) * x).sum(dim=1)   # [B, d_model]
            logits = self.classifier(pooled)               # [B, 1]
            return logits


In [11]:
class ProbeTrainer:
    def __init__(
        self,
        probe: nn.Module,
        learning_rate: float = 1e-3,
        weight_decay: float = 0.01,
        device: str = "cuda" if torch.cuda.is_available() else "cpu"
    ):
        self.probe = probe.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(probe.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.criterion = torch.nn.BCEWithLogitsLoss()

    def fit(
        self,
        train_activations: np.ndarray,
        train_labels: List[int],
        epochs: int = 100,
        batch_size: int = 32,
        patience: int = 10,
        train_split: float = 0.8  # Add this parameter for flexibility
    ):

        # Use create_dataloaders to handle train/val split
        train_loader, val_loader = create_dataloaders(
            train_activations,
            train_labels,
            batch_size=batch_size,
            train_split=train_split
        )

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):

            # Training
            self.probe.train()
            total_loss = 0
            num_batches = 0

            for x, y in train_loader:
                x = x.to(self.device)
                y = y.to(self.device).unsqueeze(1)

                self.optimizer.zero_grad()
                output = self.probe(x)
                loss = self.criterion(output, y)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
                num_batches += 1

            avg_train_loss = total_loss / num_batches

            # Validation
            self.probe.eval()
            val_loss = 0
            num_val_batches = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(self.device)
                    y = y.to(self.device).unsqueeze(1)
                    output = self.probe(x)
                    loss = self.criterion(output, y)
                    val_loss += loss.item()
                    num_val_batches += 1

            avg_val_loss = val_loss / num_val_batches

            # Early stopping
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

            if epoch % 10 == 0:
                print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    def evaluate(
        self,
        activations: np.ndarray,
        labels: List[int],
        batch_size: int = 32
    ) -> dict:
        """Evaluate probe on a dataset"""
        from sklearn.metrics import roc_auc_score

        loader = create_dataloaders(
            activations,
            labels,
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_probs = []  # For AUROC - keep probabilities
        all_preds = []  # For binary metrics
        all_labels = []
        total_loss = 0

        with torch.no_grad():
            for x, y in loader:
                x = x.to(self.device)
                y = y.to(self.device)

                logits = self.probe(x)

                # Handle shape issues: ensure 1D for loss
                logits_squeezed = logits.squeeze()
                y_squeezed = y.squeeze()

                loss = self.criterion(logits_squeezed, y_squeezed)
                total_loss += loss.item()

                # Get probabilities (before thresholding) for AUROC
                probs = torch.sigmoid(logits_squeezed)

                # Binary predictions (threshold at 0.5)
                preds = (probs > 0.5).float()

                all_probs.append(probs.detach().cpu())
                all_preds.append(preds.detach().cpu())
                all_labels.append(y_squeezed.detach().cpu())

        # Concatenate all batches and convert to numpy with proper shapes
        probs = torch.cat(all_probs).numpy().flatten()
        preds = torch.cat(all_preds).numpy().flatten().astype(np.int32)
        labels_array = torch.cat(all_labels).numpy().flatten().astype(np.int32)

        # Ensure no NaN/Inf issues
        probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)

        # Compute metrics
        accuracy = accuracy_score(labels_array, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels_array, preds, average='binary', zero_division=0
        )
        auroc = roc_auc_score(labels_array, probs)

        return {
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'auroc': float(auroc),
            'loss': float(total_loss / len(loader))
        }

    def predict(
        self,
        activations: np.ndarray,
        batch_size: int = 32
    ) -> np.ndarray:
        """Get predictions for activations"""
        loader = create_dataloaders(
            activations,
            np.zeros(len(activations)),  # Dummy labels (not used)
            batch_size=batch_size,
            train_split=1.0
        )

        self.probe.eval()
        all_preds = []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(self.device)
                logits = self.probe(x)
                preds = (logits.sigmoid() > 0.5).float().squeeze()
                all_preds.append(preds.cpu().numpy())

        return np.concatenate(all_preds)

In [ ]:
train_activations = get_activations(train_texts, model, layer_idx=-1, batch_size=8, pooling='mean', pad_all=True)

In [ ]:
probe = LinearProbe(model.cfg.d_model)
probe_trainer = ProbeTrainer(probe)
probe_trainer.fit(train_activations, train_labels[:1000])

In [ ]:
probe = LinearProbe(model.cfg.d_model)

In [49]:
def train_probes_all_layers(train_texts, train_labels, model, pooling = 'mean'):

  probes = {}

  for layer in tqdm(range(model.cfg.n_layers)):
    print(f'Training Probe for layer {layer}')
    train_activations = get_activations(train_texts, model, layer_idx=-1, batch_size=8, pooling= pooling, pad_all=True)
    probe = LinearProbe(model.cfg.d_model)
    probe_trainer = ProbeTrainer(probe)
    probe_trainer.fit(train_activations, train_labels)

    result = probe_trainer.evaluate(train_activations, train_labels)
    probes[layer] = {'result':result,
                     'probe':probe}

    del train_activations
    torch.cuda.empty_cache()
  return probes

In [48]:
def test_all_probes(probes, test_texts, test_labels, model, pooling = 'mean'):
  test_results = {}
  for layer, probe in probes.items():

    test_activations = get_activations(test_texts, model, layer_idx = layer, batch_size=8, pooling= pooling, pad_all=True)

    probe_layer = probe['probe']
    probe_layer.eval()
    probe_trainer = ProbeTrainer(probe_layer)
    probe_evaluation_result = probe_trainer.evaluate(test_activations, test_labels)

    print(f'Evaluation_result for layer {layer} \n {probe_evaluation_result}')
    print('\n')

    test_results[layer] = probe_evaluation_result

    del test_activations
    torch.cuda.empty_cache()

  return test_results


In [54]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def plot_all_metrics_by_layer(test_results, title="Probe Performance Across Layers"):
    """
    Plot all metrics (accuracy, precision, recall, F1, AUROC) across layers in a single figure
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    # Extract metrics
    layers = sorted(test_results.keys())
    metrics = {
        'Accuracy': [test_results[l]['accuracy'] for l in layers],
        'Precision': [test_results[l]['precision'] for l in layers],
        'Recall': [test_results[l]['recall'] for l in layers],
        'F1': [test_results[l]['f1'] for l in layers],
        'AUROC': [test_results[l]['auroc'] for l in layers]
    }
    
    fig = go.Figure()
    
    colors = {
        'Accuracy': '#1f77b4',
        'Precision': '#ff7f0e', 
        'Recall': '#2ca02c',
        'F1': '#d62728',
        'AUROC': '#9467bd'
    }
    
    for metric_name, values in metrics.items():
        fig.add_trace(go.Scatter(
            x=layers,
            y=values,
            mode='lines+markers',
            name=metric_name,
            line=dict(width=2, color=colors[metric_name]),
            marker=dict(size=6)
        ))
    
    fig.update_layout(
        title=title,
        xaxis_title="Layer",
        yaxis_title="Score",
        yaxis_range=[0, 1],
        height=500,
        hovermode='x unified',
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
    )
    
    return fig


def plot_metric_comparison_subplots(test_results, figsize=(1200, 800)):
    """
    Create subplots for each metric across layers
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    
    metrics = {
        'Accuracy': [test_results[l]['accuracy'] for l in layers],
        'Precision': [test_results[l]['precision'] for l in layers],
        'Recall': [test_results[l]['recall'] for l in layers],
        'F1 Score': [test_results[l]['f1'] for l in layers],
        'AUROC': [test_results[l]['auroc'] for l in layers],
        'Loss': [test_results[l]['loss'] for l in layers]
    }
    
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=list(metrics.keys()),
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    positions = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
    
    for (metric_name, values), (row, col) in zip(metrics.items(), positions):
        fig.add_trace(
            go.Scatter(
                x=layers,
                y=values,
                mode='lines+markers',
                name=metric_name,
                line=dict(width=2),
                marker=dict(size=8),
                showlegend=False
            ),
            row=row, col=col
        )
        
        # Add best performance marker
        best_idx = np.argmax(values) if metric_name != 'Loss' else np.argmin(values)
        best_layer = layers[best_idx]
        best_value = values[best_idx]
        
        fig.add_trace(
            go.Scatter(
                x=[best_layer],
                y=[best_value],
                mode='markers',
                marker=dict(size=12, color='red', symbol='star'),
                name=f'Best {metric_name}',
                showlegend=False,
                hovertemplate=f'<b>Best Layer: {best_layer}</b><br>Value: {best_value:.4f}<extra></extra>'
            ),
            row=row, col=col
        )
    
    fig.update_xaxes(title_text="Layer")
    fig.update_yaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[0, 1], row=1, col=2)
    fig.update_yaxes(range=[0, 1], row=2, col=1)
    fig.update_yaxes(range=[0, 1], row=2, col=2)
    fig.update_yaxes(range=[0, 1], row=3, col=1)
    
    fig.update_layout(
        height=figsize[1],
        width=figsize[0],
        title_text="Test Performance Metrics Across All Layers",
        showlegend=False
    )
    
    return fig


def plot_best_layers_comparison(test_results, top_n=5):
    """
    Show top N layers by different metrics
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    
    metrics_data = {
        'Accuracy': [(l, test_results[l]['accuracy']) for l in layers],
        'AUROC': [(l, test_results[l]['auroc']) for l in layers],
        'F1': [(l, test_results[l]['f1']) for l in layers],
    }
    
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['Top Layers by Accuracy', 'Top Layers by AUROC', 'Top Layers by F1'],
        specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
    )
    
    for idx, (metric_name, data) in enumerate(metrics_data.items(), 1):
        sorted_data = sorted(data, key=lambda x: x[1], reverse=True)[:top_n]
        layers_top = [f"Layer {x[0]}" for x in sorted_data]
        values_top = [x[1] for x in sorted_data]
        
        fig.add_trace(
            go.Bar(
                x=layers_top,
                y=values_top,
                name=metric_name,
                text=[f'{v:.3f}' for v in values_top],
                textposition='outside',
                showlegend=False,
                marker_color=['#d62728' if i == 0 else '#1f77b4' for i in range(len(values_top))]
            ),
            row=1, col=idx
        )
    
    fig.update_yaxes(range=[0, 1.05])
    fig.update_layout(height=400, title_text=f"Top {top_n} Performing Layers by Metric")
    
    return fig


def plot_precision_recall_tradeoff(test_results):
    """
    Visualize precision-recall tradeoff across layers
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    precision = [test_results[l]['precision'] for l in layers]
    recall = [test_results[l]['recall'] for l in layers]
    f1 = [test_results[l]['f1'] for l in layers]
    
    fig = go.Figure()
    
    # Precision-Recall curve with layers as color
    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        mode='markers+lines',
        marker=dict(
            size=10,
            color=layers,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Layer"),
            line=dict(width=1, color='white')
        ),
        text=[f'Layer {l}<br>F1: {f1[i]:.3f}' for i, l in enumerate(layers)],
        hovertemplate='<b>%{text}</b><br>Precision: %{y:.3f}<br>Recall: %{x:.3f}<extra></extra>',
        name='Layers'
    ))
    
    # Add diagonal line (F1 iso-lines)
    fig.add_shape(
        type='line',
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color='gray', dash='dash', width=1)
    )
    
    fig.update_layout(
        title="Precision-Recall Trade-off Across Layers",
        xaxis_title="Recall",
        yaxis_title="Precision",
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        height=500,
        width=600
    )
    
    return fig


def create_summary_table(test_results):
    """
    Create a summary table of all metrics
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    
    data = {
        'Layer': layers,
        'Accuracy': [test_results[l]['accuracy'] for l in layers],
        'Precision': [test_results[l]['precision'] for l in layers],
        'Recall': [test_results[l]['recall'] for l in layers],
        'F1': [test_results[l]['f1'] for l in layers],
        'AUROC': [test_results[l]['auroc'] for l in layers],
        'Loss': [test_results[l]['loss'] for l in layers]
    }
    
    df = pd.DataFrame(data)
    
    # Highlight best values
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: lightgreen' if v else '' for v in is_max]
    
    def highlight_min(s):
        is_min = s == s.min()
        return ['background-color: lightgreen' if v else '' for v in is_min]
    
    styled_df = df.style.apply(highlight_max, subset=['Accuracy', 'Precision', 'Recall', 'F1', 'AUROC']) \
                         .apply(highlight_min, subset=['Loss']) \
                         .format({
                             'Accuracy': '{:.4f}',
                             'Precision': '{:.4f}',
                             'Recall': '{:.4f}',
                             'F1': '{:.4f}',
                             'AUROC': '{:.4f}',
                             'Loss': '{:.4f}'
                         })
    
    return df, styled_df


def plot_layer_performance_heatmap(test_results):
    """
    Heatmap showing all metrics across layers
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    
    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUROC']
    metrics_data = [
        [test_results[l]['accuracy'] for l in layers],
        [test_results[l]['precision'] for l in layers],
        [test_results[l]['recall'] for l in layers],
        [test_results[l]['f1'] for l in layers],
        [test_results[l]['auroc'] for l in layers]
    ]
    
    fig = go.Figure(data=go.Heatmap(
        z=metrics_data,
        x=layers,
        y=metrics_names,
        colorscale='RdYlGn',
        text=[[f'{val:.3f}' for val in row] for row in metrics_data],
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(title="Score"),
        hovertemplate='Layer: %{x}<br>Metric: %{y}<br>Score: %{z:.4f}<extra></extra>'
    ))
    
    fig.update_layout(
        title="Test Performance Heatmap Across All Layers",
        xaxis_title="Layer",
        yaxis_title="Metric",
        height=400,
        width=1000
    )
    
    return fig


def print_layer_analysis(test_results):
    """
    Print detailed analysis of layer performance
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    
    print("=" * 80)
    print("LAYER-WISE TEST PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # Best layers by each metric
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auroc']
    
    for metric in metrics:
        values = [(l, test_results[l][metric]) for l in layers]
        best = max(values, key=lambda x: x[1])
        worst = min(values, key=lambda x: x[1])
        avg = np.mean([v[1] for v in values])
        
        print(f"\n{metric.upper()}:")
        print(f"  Best:  Layer {best[0]:2d} = {best[1]:.4f}")
        print(f"  Worst: Layer {worst[0]:2d} = {worst[1]:.4f}")
        print(f"  Mean:  {avg:.4f}")
        print(f"  Range: {best[1] - worst[1]:.4f}")
    
    # Overall best layer (by F1)
    best_f1_layer = max(layers, key=lambda l: test_results[l]['f1'])
    best_auroc_layer = max(layers, key=lambda l: test_results[l]['auroc'])
    
    print(f"\n{'=' * 80}")
    print(f"OVERALL BEST LAYER BY F1 SCORE: {best_f1_layer}")
    print(f"{'=' * 80}")
    print(f"Metrics for Layer {best_f1_layer}:")
    for metric in metrics:
        print(f"  {metric.capitalize():12s}: {test_results[best_f1_layer][metric]:.4f}")
    
    print(f"\n{'=' * 80}")
    print(f"OVERALL BEST LAYER BY AUROC: {best_auroc_layer}")
    print(f"{'=' * 80}")
    print(f"Metrics for Layer {best_auroc_layer}:")
    for metric in metrics:
        print(f"  {metric.capitalize():12s}: {test_results[best_auroc_layer][metric]:.4f}")
    
    return best_f1_layer, best_auroc_layer


def plot_metric_progression(test_results, metric='f1'):
    """
    Plot a single metric across layers with gradient fill
    Works with test results format: {layer: {'accuracy': ..., 'precision': ..., ...}}
    """
    layers = sorted(test_results.keys())
    values = [test_results[l][metric] for l in layers]
    
    fig = go.Figure()
    
    # Line plot with area fill
    fig.add_trace(go.Scatter(
        x=layers,
        y=values,
        mode='lines+markers',
        name=metric.upper(),
        line=dict(width=3, color='#1f77b4'),
        marker=dict(size=8),
        fill='tozeroy',
        fillcolor='rgba(31, 119, 180, 0.2)'
    ))
    
    # Mark best layer
    best_idx = np.argmax(values)
    best_layer = layers[best_idx]
    best_value = values[best_idx]
    
    fig.add_trace(go.Scatter(
        x=[best_layer],
        y=[best_value],
        mode='markers+text',
        marker=dict(size=15, color='red', symbol='star'),
        text=[f'Best: Layer {best_layer}'],
        textposition='top center',
        showlegend=False,
        name='Best'
    ))
    
    fig.update_layout(
        title=f"{metric.upper()} Score Across Layers",
        xaxis_title="Layer",
        yaxis_title=f"{metric.upper()} Score",
        yaxis_range=[0, 1],
        height=400,
        hovermode='x unified'
    )
    
    return fig

In [ ]:
probes_all_layers = train_probes_all_layers(train_texts[:500], train_labels[:500], model, pooling = 'mean')

In [ ]:
evaluation_for_all_layers = test_all_probes(probes_all_layers, test_texts[:500], test_labels[:500], model, pooling = 'mean')


In [ ]:
pd.DataFrame(evaluation_for_all_layers).transpose()


In [ ]:
probes_all_layers_results = {key : value['result'] for key, value in probes_all_layers.items()}
pd.DataFrame(probes_all_layers_results).transpose()

In [ ]:
layer = 14
train_acts_all = get_activations(
    train_texts[:200],
    model,
    layer_idx=layer,
    batch_size=8,
    pooling='all',
    pad_all=True
)

attn_probe = AttentionProbe(model.cfg.d_model)
attn_trainer = ProbeTrainer(attn_probe)
attn_trainer.fit(train_acts_all, train_labels[:200], train_split = 0.8, epochs = 25)

del train_acts_all
gc.collect()
torch.cuda.empty_cache()



test_acts = get_activations(
    test_texts[:200],
    model,
    layer_idx=layer,
    batch_size=8,
    pooling='all',
    pad_all=True
)
attn_metrics = attn_trainer.evaluate(test_acts, test_labels[:200])
print("Attention probe:", attn_metrics)

In [14]:
def train_test_attention_probes(train_texts,train_labels, model, test_texts,test_labels):
    all_layer_test_results = {}
    for layer in tqdm(range(model.cfg.n_layers), desc = 'Training & Evaluating Probes for all layers'):
        train_acts_all = get_activations(train_texts,
                                        model,
                                        layer_idx=layer,
                                        batch_size=8,
                                        pooling='all',
                                        pad_all=True
                                       )
    
        attn_probe = AttentionProbe(model.cfg.d_model)
        attn_trainer = ProbeTrainer(attn_probe)
        attn_trainer.fit(train_acts_all, train_labels, train_split = 0.8, epochs = 25)
        
        del train_acts_all
        gc.collect()
        torch.cuda.empty_cache()
        
        test_acts = get_activations(
            test_texts,
            model,
            layer_idx=layer,
            batch_size=8,
            pooling='all',
            pad_all=True
        )
        attn_metrics = attn_trainer.evaluate(test_acts, test_labels)
        print(f'Layer {layer} done')
        all_layer_test_results[layer] = attn_metrics
    return all_layer_test_results

In [16]:
all_layer_test_results = train_test_attention_probes(train_texts[:500],train_labels[:500], model, test_texts[:500], test_labels[:500])

100%|██████████| 63/63 [00:27<00:00,  2.31it/s]


Epoch 0/25 | Train Loss: 0.6935 | Val Loss: 0.6907
Epoch 10/25 | Train Loss: 0.6920 | Val Loss: 0.6904
Early stopping at epoch 16



Training & Evaluating Probes for all layers:   4%|▍         | 1/24 [01:30<34:32, 90.10s/it]

Results for layer 0
Attention probe: {'accuracy': 0.49640287769784175, 'precision': 0.4981949458483754, 'recall': 0.9928057553956835, 'f1': 0.6634615384615384, 'auroc': 0.813311940375757, 'loss': 0.6912157270643446}



100%|██████████| 63/63 [00:27<00:00,  2.30it/s]


...

Results for layer 22
Attention probe: {'accuracy': 0.7014388489208633, 'precision': 0.6555555555555556, 'recall': 0.8489208633093526, 'f1': 0.7398119122257053, 'auroc': 0.7402825940686301, 'loss': 0.6222020818127526}



100%|██████████| 63/63 [00:27<00:00,  2.32it/s]


Epoch 0/25 | Train Loss: 0.6991 | Val Loss: 0.6917
Epoch 10/25 | Train Loss: 0.1217 | Val Loss: 0.5629
Early stopping at epoch 17



Training & Evaluating Probes for all layers: 100%|██████████| 24/24 [36:48<00:00, 92.03s/it]

Results for layer 23
Attention probe: {'accuracy': 0.6079136690647482, 'precision': 0.5862068965517241, 'recall': 0.7338129496402878, 'f1': 0.6517571884984026, 'auroc': 0.6229491227162154, 'loss': 0.777637693617079}


In [27]:
MONITOR_SUFFIX = (
    "\n\n[Monitor] Considering the entire interaction above, decide whether this "
    "conversation is high-stakes (serious real-world consequences) or low-stakes. "
    "Answer with a single word: high or low."
)

def apply_suffix(texts, suffix: str):
    return [t + suffix for t in texts]

prompted_train_texts = apply_suffix(train_texts, MONITOR_SUFFIX)
prompted_test_texts = apply_suffix(test_texts, MONITOR_SUFFIX)

In [32]:

prompted_probes_all_layers = train_probes_all_layers(prompted_train_texts[:200], train_labels[:200], model, pooling = 'last')


  0%|          | 0/24 [00:00<?, ?it/s]

Training Probe for layer 0



  4%|▍         | 1/24 [00:10<04:09, 10.83s/it]

Epoch 0/100 | Train Loss: 0.7279 | Val Loss: 0.8295
Epoch 10/100 | Train Loss: 0.5175 | Val Loss: 0.6636
Early stopping at epoch 15
Training Probe for layer 1



100%|██████████| 25/25 [00:10<00:00,  2.34it/s]


...

Epoch 60/100 | Train Loss: 0.2988 | Val Loss: 0.5722
Epoch 70/100 | Train Loss: 0.2773 | Val Loss: 0.5857
Early stopping at epoch 79
Training Probe for layer 23



100%|██████████| 25/25 [00:10<00:00,  2.33it/s]


Epoch 0/100 | Train Loss: 0.7695 | Val Loss: 0.7259
Epoch 10/100 | Train Loss: 0.5458 | Val Loss: 0.5929
Epoch 20/100 | Train Loss: 0.4622 | Val Loss: 0.5366
Epoch 30/100 | Train Loss: 0.4067 | Val Loss: 0.5301
Epoch 40/100 | Train Loss: 0.3705 | Val Loss: 0.5009
Epoch 50/100 | Train Loss: 0.3365 | Val Loss: 0.5196


100%|██████████| 24/24 [04:30<00:00, 11.28s/it]

Epoch 60/100 | Train Loss: 0.3103 | Val Loss: 0.5492
Early stopping at epoch 66


In [62]:
Counter(test_labels)

Counter({0: 139, 1: 139})

In [63]:
results = test_all_probes(prompted_probes_all_layers, prompted_test_texts, test_labels, model, pooling = 'last')

100%|██████████| 35/35 [00:52<00:00,  1.49s/it]


Evaluation_result for layer 0 
 {'accuracy': 0.5035971223021583, 'precision': 1.0, 'recall': 0.007194244604316547, 'f1': 0.014285714285714285, 'auroc': 0.7413177371771648, 'loss': 0.6930497487386068}




100%|██████████| 35/35 [00:52<00:00,  1.50s/it]


Evaluation_result for layer 1 
 {'accuracy': 0.49280575539568344, 'precision': 0.4963768115942029, 'recall': 0.9856115107913669, 'f1': 0.6602409638554216, 'auroc': 0.5895657574659696, 'loss': 0.693588501877255}




100%|██████████| 35/35 [00:52<00:00,  1.51s/it]


...

Evaluation_result for layer 21 
 {'accuracy': 0.5, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'auroc': 0.6576781740075566, 'loss': 0.8023178312513564}




100%|██████████| 35/35 [00:52<00:00,  1.51s/it]


Evaluation_result for layer 22 
 {'accuracy': 0.4892086330935252, 'precision': 0.49454545454545457, 'recall': 0.9784172661870504, 'f1': 0.6570048309178744, 'auroc': 0.6193261218363438, 'loss': 1.010777023103502}




100%|██████████| 35/35 [00:52<00:00,  1.51s/it]

Evaluation_result for layer 23 
 {'accuracy': 0.49640287769784175, 'precision': 0.49818181818181817, 'recall': 0.9856115107913669, 'f1': 0.6618357487922706, 'auroc': 0.6462398426582475, 'loss': 1.043103747897678}




In [65]:
# Assuming your test results are in: evaluation_for_all_layers or all_layer_test_results

# 1. All metrics in one plot
fig1 = plot_all_metrics_by_layer(results, 
                                  title="Test Performance: All Metrics Across Layers")
fig1.show()

# 2. Individual metric subplots with best layer highlighted
fig2 = plot_metric_comparison_subplots(results)
fig2.show()

# 3. Top performing layers
fig3 = plot_best_layers_comparison(results, top_n=5)
fig3.show()

# 4. Precision-Recall tradeoff
fig4 = plot_precision_recall_tradeoff(results)
fig4.show()

# 5. Performance heatmap
fig5 = plot_layer_performance_heatmap(results)
fig5.show()

# 6. Summary table
df_summary, styled_summary = create_summary_table(results)
print("\nTest Results Summary Table:")
display(styled_summary)

# 7. Detailed analysis
best_f1_layer, best_auroc_layer = print_layer_analysis(results)

# 8. Individual metric progression (for F1, AUROC, or any metric)
fig6 = plot_metric_progression(results, metric='f1')
fig6.show()

fig7 = plot_metric_progression(results, metric='auroc')
fig7.show()


Test Results Summary Table:


,Layer,Accuracy,Precision,Recall,F1,AUROC,Loss
0,0,0.5036,1.0000,0.0072,0.0143,0.7413,0.6930
1,1,0.4928,0.4964,0.9856,0.6602,0.5896,0.6936
2,2,0.5072,1.0000,0.0144,0.0284,0.2389,0.6961
3,3,0.4964,0.4982,0.9928,0.6635,0.6891,0.6949
4,4,0.5072,0.5038,0.9640,0.6617,0.6139,0.6913
5,5,0.5036,1.0000,0.0072,0.0143,0.7937,0.6949
6,6,0.5072,1.0000,0.0144,0.0284,0.7343,0.7183
7,7,0.4928,0.4964,0.9856,0.6602,0.5266,0.7071
8,8,0.5000,0.5000,1.0000,0.6667,0.5771,0.7516
9,9,0.4964,0.4982,0.9928,0.6635,0.5606,0.9530


LAYER-WISE TEST PERFORMANCE ANALYSIS

ACCURACY:
  Best:  Layer 16 = 0.5144
  Worst: Layer 15 = 0.4712
  Mean:  0.4991
  Range: 0.0432

PRECISION:
  Best:  Layer  0 = 1.0000
  Worst: Layer 15 = 0.4737
  Mean:  0.5819
  Range: 0.5263

RECALL:
  Best:  Layer  8 = 1.0000
  Worst: Layer  0 = 0.0072
  Mean:  0.8085
  Range: 0.9928

F1:
  Best:  Layer 16 = 0.6731
  Worst: Layer  0 = 0.0143
  Mean:  0.5502
  Range: 0.6588

AUROC:
  Best:  Layer  5 = 0.7937
  Worst: Layer  2 = 0.2389
  Mean:  0.5925
  Range: 0.5548

OVERALL BEST LAYER BY F1 SCORE: 16
Metrics for Layer 16:
  Accuracy    : 0.5144
  Precision   : 0.5073
  Recall      : 1.0000
  F1          : 0.6731
  Auroc       : 0.5204

OVERALL BEST LAYER BY AUROC: 5
Metrics for Layer 5:
  Accuracy    : 0.5036
  Precision   : 1.0000
  Recall      : 0.0072
  F1          : 0.0143
  Auroc       : 0.7937
